## 0. Commun fuctions
### Create experiment output as a csv

In [23]:
import csv
import os
import datetime

def log_experiment_csv(csv_file,version_name, input ,start_time, end_time, score, parameters):
  
    duration_seconds = end_time - start_time
    file_name = csv_file
    
    # Prepare data fields
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    
    # Define the row data
    row = {
        "Timestamp": timestamp,
        "Version": version_name,
        "Input": input,
        "Score": score,
        "Duration_Sec": round(duration_seconds, 2),
        "Parameters": parameters
    }
    
    file_exists = os.path.isfile(file_name)
    
    # Write to CSV
    with open(file_name, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        
        # If file is new, write the header first
        if not file_exists:
            writer.writeheader()
            
        writer.writerow(row)

### Load input dataset

In [24]:
from pathlib import Path

def load_problem(file_in):
    pizza_list=[]
    teams={}

    with open(file_in) as file:
            for line_index, line in enumerate(file):
                actual_line = line.strip().split()
                if line_index == 0:
                    teams = {
                        2: int(actual_line[1]),
                        3: int(actual_line[2]),
                        4: int(actual_line[3])
                    }
                else:
                    pizza = {
                    "id": line_index - 1,
                    "n_ingredients": int(actual_line[0]),
                    "ingredients": actual_line[1:]
                }
                    pizza_list.append(pizza)
    return(teams,pizza_list)
    

### Assign a score to deliveries 

In [25]:
def score(pizzas,deliveries):
    score_list=[]
    for i in range(len(deliveries)):
        delivered_pizzas=deliveries[i][1]
        ingredients=[]
        for id in delivered_pizzas:
            ingredient_list=pizzas[id]['ingredients']
            for ingredient in ingredient_list:
                ingredients.append(ingredient)
        unique_ingredients=set(ingredients)
        score_list.append(len(unique_ingredients)**2)
        #print(score_list)
    if len(deliveries)==1:
        total_score=score_list[0]
    else:
        total_score=sum(score_list)
    return total_score

## 1. Greedy function to generate a solution to the problem

In [26]:
import random

def solve_pizza_problem(pizzas, t2, t3, t4,sort_set=True):
    # Sort pizzas by number of ingredients (descending) to start with 'richer' options
    pizzas.sort(key=lambda x: len(x['ingredients']), reverse=True)
    teams = [(4, t4), (3, t3), (2, t2)]
    if not sort_set:
        random.shuffle(pizzas)
        random.shuffle(teams)
    
    deliveries = []
    used_pizzas = [False] * len(pizzas)
    
    # Process teams from largest to smallest (4 -> 3 -> 2) 
    # because squares of larger numbers yield higher scores.
    for team_size, team_count in teams:
        for _ in range(team_count):
            current_team_pizzas = []
            current_ingredients = set()
            
            # Fill the team requirements
            for _ in range(team_size):
                best_pizza_idx = -1
                max_new_ingredients = -1
                
                # Look for the pizza that adds the most value
                for i in range(len(pizzas)):
                    if not used_pizzas[i]:
                        # Calculate how many NEW ingredients this pizza adds
                        new_count = len(set(pizzas[i]['ingredients']) - current_ingredients)
                        
                        if new_count > max_new_ingredients:
                            max_new_ingredients = new_count
                            best_pizza_idx = i
                        
                        # Optimization: if a pizza adds all its ingredients as new, 
                        # it's a strong candidate; we can break early in large datasets.
                
                if best_pizza_idx != -1:
                    used_pizzas[best_pizza_idx] = True
                    current_team_pizzas.append(pizzas[best_pizza_idx]['id'])
                    current_ingredients.update(pizzas[best_pizza_idx]['ingredients'])
                else:
                    # Not enough pizzas left to fill this team
                    break
            
            if len(current_team_pizzas) == team_size:
                deliveries.append((team_size, current_team_pizzas))
            else:
                # Backtrack: if team wasn't filled, mark pizzas as available again
                for p_id in current_team_pizzas:
                    used_pizzas[p_id] = False
                    
    return deliveries

## 2. GA Algorithm

In [32]:
import random
import collections
import numpy as np
import time
import matplotlib.pyplot as plt


# --- Pre-processing ---
# Map ingredients to bitsets for O(1) union operations
def get_pizza_bitsets(pizzas):
    all_ingredients = sorted(list(set(ing for p in pizzas for ing in p['ingredients'])))
    ing_map = {ing: i for i, ing in enumerate(all_ingredients)}
    
    pizza_bitsets = []
    for p in pizzas:
        bits = 0
        for ing in p['ingredients']:
            bits |= (1 << ing_map[ing])
        pizza_bitsets.append(bits)
    return pizza_bitsets

# --- Fitness Function ---
def calculate_fitness(chromosome, pizza_bitsets):
    total_score = 0
    for team_size, pizza_ids in chromosome:
        combined_bits = 0
        for p_id in pizza_ids:
            combined_bits |= pizza_bitsets[p_id]
        
        # Score = (number of unique ingredients)^2 [cite: 106, 109]
        unique_count = bin(combined_bits).count('1')
        total_score += unique_count**2
    return total_score

# --- Crossover ---
def crossover(parent1, parent2, pizza_count):
    # Take half of deliveries from Parent 1
    child = parent1[:len(parent1)//2]
    used_pizzas = {p_id for _, p_list in child for p_id in p_list}
    
    # Fill remaining from Parent 2 if pizzas are available
    for team_size, p_list in parent2:
        if all(p_id not in used_pizzas for p_id in p_list):
            child.append((team_size, p_list))
            used_pizzas.update(p_list)
    return child

# --- Mutation ---
def mutate(chromosome, pizza_bitsets, available_pizzas):
    if len(chromosome) < 2: return chromosome
    
    # 1. Swap Mutation: Swap one pizza between two deliveries of same size
    idx1, idx2 = random.sample(range(len(chromosome)), 2)
    if chromosome[idx1][0] == chromosome[idx2][0]:
        p_list1, p_list2 = list(chromosome[idx1][1]), list(chromosome[idx2][1])
        i1, i2 = random.randint(0, len(p_list1)-1), random.randint(0, len(p_list2)-1)
        
        p_list1[i1], p_list2[i2] = p_list2[i2], p_list1[i1]
        chromosome[idx1] = (chromosome[idx1][0], p_list1)
        chromosome[idx2] = (chromosome[idx2][0], p_list2)
        
    return chromosome


def run_genetic_pizza(pizzas, t2, t3, t4, generations=100, pop_size=20,sort_set=False,shuffled=False):
    # Start the total execution timer
    start_total = time.time()
    
    pizza_bits = get_pizza_bitsets(pizzas)
    history = [] 
    
    print(f"--- Starting GA for {generations} generations ---")
    
    # 1. Initialization Time
    start_init = time.time()
    population = []
    
    if shuffled:
        for i in range(pop_size):
            shuffled_pizzas = random.sample(pizzas, len(pizzas))
            population.append(solve_pizza_problem(shuffled_pizzas, t2, t3, t4,True))
    else:
        delivery = solve_pizza_problem(pizzas, t2, t3, t4,False)
        for i in range(pop_size):
            population.append(delivery)        
               
    
    end_init = time.time()
    
    print(f"Initialization took: {end_init - start_init:.2f} seconds")

    # 2. Evolution Loop
    for gen in range(generations):
        start_gen = time.time()
        
        # Sort by fitness
        population.sort(key=lambda c: calculate_fitness(c, pizza_bits), reverse=True)
        
        current_best_score = calculate_fitness(population[0], pizza_bits)
        history.append(current_best_score)
        
        # Elitism
        new_gen = population[:2]
        
        # Breeding
        while len(new_gen) < pop_size:
            selection_pool = population[:max(2, pop_size // 4)]
            p1, p2 = random.sample(selection_pool, 2)
            child = crossover(p1, p2, len(pizzas))
            child = mutate(child, pizza_bits, len(pizzas))
            new_gen.append(child)
            
        population = new_gen
        end_gen = time.time()
        
        # Print progress and time per generation every 10 generations
        if gen % 10 == 0:
            gen_time = end_gen - start_gen
            print(f"Gen {gen} | Best: {current_best_score} | Time: {gen_time:.4f}s")

    # Final Stats
    end_total = time.time()
    total_duration = end_total - start_total
    
    print("-" * 30)
    print(f"Total Execution Time: {total_duration:.2f} seconds")
    print(f"Average Time per Generation: {total_duration/generations:.4f} seconds")
    print("-" * 30)

    # Plotting the results
    #plt.plot(history)
    #plt.title(f"GA Progress (Total Time: {total_duration:.2f}s)")
    #plt.xlabel("Generation")
    #plt.ylabel("Total Score")
    #plt.show()
    
    return population[0]

## 3. Tabu Search

In [30]:
import time
import copy

def tabu_search(pizzas, t2, t3, t4, iterations=500, tabu_size=50):

    pizza_dict = {p["id"]:p for p in pizzas}

    current_solution = solve_pizza_problem(pizzas, t2, t3, t4)
    best_solution = copy.deepcopy(current_solution)

    best_score = score(pizzas,best_solution)

    tabu_list=[]

    all_pizzas=set(pizza_dict.keys())

    for it in range(iterations):

        candidate = copy.deepcopy(current_solution)

        move_type = random.choice(["swap","replace"])

        if move_type=="swap" and len(candidate)>=2:

            d1,d2=random.sample(range(len(candidate)),2)

            team1,pizzas1=candidate[d1]
            team2,pizzas2=candidate[d2]

            i=random.randrange(len(pizzas1))
            j=random.randrange(len(pizzas2))

            move=(pizzas1[i],pizzas2[j])

            pizzas1[i],pizzas2[j]=pizzas2[j],pizzas1[i]

            candidate[d1]=(team1,pizzas1)
            candidate[d2]=(team2,pizzas2)

        else:

            used=set(p for _,plist in candidate for p in plist)
            unused=list(all_pizzas-used)

            if not unused:
                continue

            d=random.randrange(len(candidate))
            team,pizza_ids=candidate[d]

            idx=random.randrange(len(pizza_ids))

            new_pizza=random.choice(unused)

            move=(pizza_ids[idx],new_pizza)

            pizza_ids[idx]=new_pizza

            candidate[d]=(team,pizza_ids)

        candidate_score=score(pizzas,candidate)

        if move in tabu_list and candidate_score<=best_score:
            continue

        current_solution=candidate

        if candidate_score>best_score:

            best_solution=copy.deepcopy(candidate)
            best_score=candidate_score

        tabu_list.append(move)

        if len(tabu_list)>tabu_size:
            tabu_list.pop(0)

        if it % max(1, iterations//10) == 0:
            print("Iteration",it,"Best score:",best_score)

    return best_solution



# Output Function


def output_write(output_path,deliveries):

    with open(output_path, "w") as file:

        file.write(str(len(deliveries)))
        file.write("\n")

        for team_size, pizza_id in deliveries:

            line = " ".join(map(str, [team_size] + pizza_id))
            file.write(line + "\n")



# Paths


from pathlib import Path

data_folder = Path("../data")
output_folder = Path("../out")
direc=Path().absolute()

input_path= {
    "a": data_folder / "a_example",
    "b": data_folder / "b_little_bit_of_everything.in",
    "c": data_folder / "c_many_ingredients.in",
    "d": data_folder / "d_many_pizzas.in",
    "e": data_folder / "e_many_teams.in"
}

output_path = {
    "a": output_folder / "a_example.txt",
    "b": output_folder / "b_little_bit_of_everything.txt",
    "c": output_folder / "c_many_ingredients.txt",
    "d": output_folder / "d_many_pizzas.txt",
    "e": output_folder / "e_many_teams.txt"
}



# 6. Parameter Grid


parameter_grid = [
    {"iterations": 500, "tabu_size": 25},
    {"iterations": 500, "tabu_size": 50},
    {"iterations": 1000, "tabu_size": 50},
    {"iterations": 1000, "tabu_size": 100}
]



# 7. Execution Pipeline


results = []

for data in ["a","b","c","d","e"]:

    print("\n==============================")
    print("Running dataset:", data)
    print("==============================")

    pizza_list=[]
    teams={}

    input_loc=input_path[data]
    output_loc=output_path[data]

    with open(input_loc) as file:

        for line_index, line in enumerate(file):

            actual_line = line.strip().split()

            if line_index == 0:

                teams = {
                    2: int(actual_line[1]),
                    3: int(actual_line[2]),
                    4: int(actual_line[3])
                }

            else:

                pizza = {
                    "id": line_index - 1,
                    "ingredients": set(actual_line[1:])
                }

                pizza_list.append(pizza)

    print("Loaded pizzas:", len(pizza_list))


    # -----------------------
    # Greedy Baseline
    # -----------------------

    start_time = time.time()

    deliveries = solve_pizza_problem(pizza_list, teams[2], teams[3], teams[4])

    initial_score = score(pizza_list, deliveries)

    greedy_time = time.time() - start_time

    print("Initial greedy score:", initial_score)

    results.append({
        "dataset": data,
        "algorithm": "Greedy",
        "iterations": 0,
        "tabu_size": 0,
        "score": initial_score,
        "runtime_seconds": greedy_time
    })


    # -----------------------
    # Tabu Search
    # -----------------------

    for params in parameter_grid:

        print("\nTesting parameters:", params)

        start_time = time.time()

        best_solution = tabu_search(
            pizza_list,
            teams[2],
            teams[3],
            teams[4],
            iterations=params["iterations"],
            tabu_size=params["tabu_size"]
        )

        final_score = score(pizza_list, best_solution)

        tabu_time = time.time() - start_time

        print("Final score:", final_score)

        results.append({
            "dataset": data,
            "algorithm": "Tabu Search",
            "iterations": params["iterations"],
            "tabu_size": params["tabu_size"],
            "score": final_score,
            "runtime_seconds": tabu_time,
            "improvement_vs_greedy": final_score - initial_score
        })

    output_write(output_loc, best_solution)


# =========================================================
# 8. Export Results
# =========================================================

results_df = pd.DataFrame(results)

excel_output = output_folder / "experiment_results.xlsx"

score_table = results_df.pivot_table(
    index="dataset",
    columns="algorithm",
    values="score",
    aggfunc="max"
)

time_table = results_df.pivot_table(
    index="dataset",
    columns="algorithm",
    values="runtime_seconds",
    aggfunc="mean"
)

parameter_table = results_df.pivot_table(
    index=["iterations", "tabu_size"],
    values="score",
    aggfunc="mean"
)

with pd.ExcelWriter(excel_output, engine="xlsxwriter") as writer:

    results_df.to_excel(writer, sheet_name="Results", index=False)

    parameter_table.to_excel(writer, sheet_name="Parameter Analysis")
    score_table.to_excel(writer, sheet_name="Score Table")
    time_table.to_excel(writer, sheet_name="Runtime Table")

print("\nResults exported to:", excel_output)


Running dataset: a
Loaded pizzas: 5
Initial greedy score: 49

Testing parameters: {'iterations': 500, 'tabu_size': 25}


AttributeError: 'builtin_function_or_method' object has no attribute 'choice'

## 4. Simulated Annealing

In [34]:
from random import random, randrange, choice
from math import exp

def get_neighbour(deliveries,pizzas):
    new_deliveries = [ (t, p.copy()) for t,p in deliveries ]

    team_idx = randrange(len(new_deliveries))
    team_size, pizza_ids = new_deliveries[team_idx]

    all_used = {p for _,plist in new_deliveries for p in plist}
    unused = [p['id'] for p in pizzas if p['id'] not in all_used]

    if not unused:
        return new_deliveries

    replace_idx = randrange(team_size)
    pizza_ids[replace_idx] = choice(unused)

    new_deliveries[team_idx] = (team_size, pizza_ids)

    return new_deliveries

def simulated_annealing(pizzas, current_solution, current_score,
                        T_max=1000,
                        T_min=0.1,
                        cooling_rate=0.95,
                        iterations_per_temp=5):

    # original parameters: T_max=1000 T_min=0.1 cooling_rate=0.95 iterations_per_temp=100

    start_total = time.time()

    history=[]
    count=0
    T_list=[]

    best_solution = current_solution
    best_score = current_score

    T = T_max

    print(f"Start simulated annealing with maximum temperature= {T_max}; minimum temperature= {T_min}; cooling rate= {cooling_rate}; number of iterations per temperature= {iterations_per_temp} \n")

    while T > T_min:

        start_temp=time.time()

        for _ in range(iterations_per_temp):

            #start_iter=time.time()
            history.append(best_score)
            T_list.append(T)
            neighbour = get_neighbour(current_solution, pizzas)
            neighbour_score = score(pizzas, neighbour)

            delta = neighbour_score - current_score
            

            if delta > 0:
                accept = True
            else:
                accept = random() < exp(delta / T)

            if accept:
                current_solution = neighbour
                current_score = neighbour_score

                if current_score > best_score:
                    best_solution = current_solution
                    best_score = current_score
            #end_iter=time.time()
        count+=1
        T *= cooling_rate
        end_temp=time.time()
        if count % 10: 
            temp_time = end_temp - start_temp
            print(f"Temperature {T} | Best: {best_score} | Time: {temp_time:.4f}s")

    end_total=time.time()
    total_duration = end_total - start_total

    print("-" * 30)
    print(f"Total Execution Time: {total_duration:.2f} seconds")
    print(f"Average Time per Temperature: {total_duration*iterations_per_temp/len(T_list):.4f} seconds")
    print("-" * 30)

    plt.plot(T_list,history)
    plt.title(f"Simulated Annealing Progress (Total Time: {total_duration:.2f}s)")
    plt.xlabel("Temperature")
    plt.ylabel("Total Score")
    plt.show()

    return best_solution, best_score

## 5. Run Experiments

In [35]:
from pathlib import Path

#Set result output file
num_trials = 30

data_folder = Path("../data")
output_folder = Path("../out")

input_path= {
    "a": data_folder / "a_example",
    "b": data_folder / "b_little_bit_of_everything.in",
    "c": data_folder / "c_many_ingredients.in",
    "d": data_folder / "d_many_pizzas.in",
    "e": data_folder / "e_many_teams.in"
}

csv_file = output_folder / "results.csv"



# run 30 times 
for i in range(1, num_trials + 1):
    #Run  11 | run_genetic_pizza | a_example | POP =100, GEN = 100
    version_name = "run_genetic_pizza"
    file_location = input_path["a"]
    parameters = "GEN=100 POP=100"
    start_time =  time.time()

    (teams,pizza_list) = load_problem(file_location)
    deliveries = run_genetic_pizza(pizza_list,teams[2], teams[3], teams[4],generations=100, pop_size=100)
    end_time =  time.time()
    e_score = score(pizza_list,deliveries)
    log_experiment_csv(csv_file,version_name,os.path.basename(file_location),start_time,end_time,e_score,parameters)
    
    

--- Starting GA for 100 generations ---


AttributeError: 'builtin_function_or_method' object has no attribute 'shuffle'